# Import packages and data

In [1]:
# %pip install openpyxl
# %pip install thefuzz
import pandas as pd
import numpy as np
import re
from collections import Counter
from thefuzz import fuzz


In [2]:
# Pre-processing function to clean and normalize phone numbers to 62 format
def normalize_to_62(phone_series):
   # 1. Convert to string and strip spaces
    cleaned = phone_series.astype(str).str.strip()
    
    # 2. CRITICAL FIX: Fix the float '.0' issue FIRST
    # (Looks specifically for a dot followed by a zero at the end of the string)
    cleaned = cleaned.str.replace(r'\.0$', '', regex=True)
    
    # 3. NOW it is safe to remove all other non-digit characters
    cleaned = cleaned.str.replace(r'\D', '', regex=True)
    def fix_prefix(x):
        # 🛡️ Force x to be a string just in case pandas sneaks a float in!
        x_str = str(x).strip()
        
        # Ignore empty values
        if x_str.lower() in ['nan', 'none', '<na>', '']:
            return np.nan 
            
        # If the number is too short to be valid, return it as is (or you could choose to return NaN)
        if len(x_str) < 9:
            return x_str # Return the short number exactly as it is
        
        # If it starts with '0', replace '0' with '62'
        if x_str.startswith('0'):
            return '62' + x_str[1:]
            
        # If it starts with '8', prepend '62'
        elif x_str.startswith('8'):
            return '62' + x_str
            
        return x_str

    return cleaned.apply(fix_prefix)

In [3]:
# pre-processing function to clean and normalize names by removing punctuation and converting to lowercase

# ==========================================
# 1. GENERATE DYNAMIC STOP WORDS
# ==========================================
def generate_dynamic_stop_words(df: pd.DataFrame, columns: list, top_n: int = 15) -> set:
    """
    Scans the master data to find the most frequent junk words (e.g., Toko, CV, PT).
    """
    all_text = pd.Series(dtype=str)
    
    # Combine text from all specified columns safely
    for col in columns:
        if col in df.columns:
            all_text = pd.concat([all_text, df[col].dropna().astype(str)])

    # Lowercase everything and strip out all punctuation
    clean_text = all_text.str.lower().apply(lambda x: re.sub(r'[^\w\s]', '', x))

    # Split into individual words and count them
    all_words = " ".join(clean_text).split()
    word_counts = Counter(all_words)

    # Extract just the top words into a Set
    stop_words = set([word for word, count in word_counts.most_common(top_n)])
    
    # Print them out so you can manually verify them!
    print(f"[DEBUG] Top {top_n} Dynamic Stop Words Generated:")
    for word, count in word_counts.most_common(top_n):
        print(f"  - {word}: {count} times")
        
    return stop_words

# ==========================================
# 2. CREATE CLEAN NAME SETS
# ==========================================
def create_name_set(name, stop_words: set) -> set:
    """
    Takes a raw name, cleans it, removes stop words, and returns a Set of words.
    """
    # Handle empty or NaN values safely
    if pd.isnull(name) or str(name).strip() == "":
        return set()
        
    name_str = str(name).lower()
    
    # Remove punctuation (turns "Toko Murah!" into "toko murah")
    name_clean = re.sub(r'[^\w\s]', '', name_str)
    
    # Split into words and keep only the ones that are NOT in the stop words list
    final_words = [word for word in name_clean.split() if word not in stop_words]
    
    return set(final_words)

In [4]:
# Entity Resolution Function

def run_entity_resolution(data_clean: pd.DataFrame, existing_data: pd.DataFrame) -> pd.DataFrame:
    matched_records = []
    
    print(f"[DEBUG] Starting Entity Resolution for {len(data_clean)} scraped records...")

    for index, scraped_row in data_clean.iterrows():
        scraped_phone = str(scraped_row.get('phone', '')).strip()
        scraped_name_set = scraped_row.get('author_name_set', set())
        word_count = len(scraped_name_set)
        
        has_valid_phone = scraped_phone.lower() not in ['nan', 'none', '', '<na>']
        match_found = False

        # =========================================================
        # CHECK 1: PHONE MATCH (Absolute Certainty)
        # =========================================================
        if has_valid_phone:
            # Look for exact phone match in either master column
            phone_matches = existing_data[
                (existing_data['nomor_telepon'].astype(str).str.strip() == scraped_phone) |
                (existing_data['nomor_whatsapp'].astype(str).str.strip() == scraped_phone)
            ]
            
            if not phone_matches.empty:
                for _, exist_row in phone_matches.iterrows():
                    # Merge the data dictionaries together
                    combined_row = {**scraped_row.to_dict(), **exist_row.to_dict()}
                    combined_row['comparison_status'] = 'Registered'
                    matched_records.append(combined_row)
                
                continue # Skip the name check, move to the next scraped lead

        # =========================================================
        # CHECK 2: NAME MATCH (Fuzzy Logic)
        # =========================================================
        if word_count > 0:
            matched_master_indices = []
            
            # Reconstruct the cleaned scraped name into a normal string for 2+ word checking
            scraped_clean_str = " ".join(scraped_name_set)
            
            # Scan through every master record
            for exist_idx, exist_row in existing_data.iterrows():
                # Combine legal and commercial word sets to check everything at once
                master_set_legal = exist_row.get('nama_usaha_set', set())
                master_set_komersial = exist_row.get('nama_komersial_set', set())
                combined_master_set = master_set_legal.union(master_set_komersial)
                
                if not combined_master_set:
                    continue # Skip if master data has no usable name words
                    
                best_score = 0
                
                if word_count == 1:
                    # RULE: 1-Word Fuzzy Match. 
                    # Does this single scraped word closely match ANY single word in the master data?
                    scraped_word = list(scraped_name_set)[0]
                    for m_word in combined_master_set:
                        score = fuzz.ratio(scraped_word, m_word)
                        if score > best_score:
                            best_score = score
                            
                    threshold = 85 # 85% similarity handles typos like "Ilyass" vs "Ilyas"
                    
                else:
                    # RULE: 2+ Word Fuzzy Match.
                    # Ignore order and extra words. "ilyas murah" matches "toko sepatu murah ilyas"
                    master_clean_str_1 = " ".join(master_set_legal)
                    master_clean_str_2 = " ".join(master_set_komersial)
                    
                    score1 = fuzz.token_set_ratio(scraped_clean_str, master_clean_str_1)
                    score2 = fuzz.token_set_ratio(scraped_clean_str, master_clean_str_2)
                    best_score = max(score1, score2)
                    
                    threshold = 80 # Slightly lower threshold since token_set_ratio is strict on word overlap
                
                # If it passes the threshold, record the master row index!
                if best_score >= threshold:
                    matched_master_indices.append(exist_idx)

            # --- Process the Name Matches ---
            if len(matched_master_indices) > 0:
                # If a 1-word name matched too many businesses, flag it for manual review
                if word_count == 1 and len(matched_master_indices) > 3:
                    status = 'Name (Multiple - Verify)'
                else:
                    status = 'Name'
                
                for exist_idx in matched_master_indices:
                    exist_row = existing_data.loc[exist_idx]
                    combined_row = {**scraped_row.to_dict(), **exist_row.to_dict()}
                    combined_row['comparison_status'] = status
                    matched_records.append(combined_row)
                    
                continue # Move to next scraped lead

        # =========================================================
        # CHECK 3: NO MATCH (Brand New Lead)
        # =========================================================
        combined_row = scraped_row.to_dict()
        combined_row['comparison_status'] = 'None'
        matched_records.append(combined_row)

    # Convert everything back to a beautiful DataFrame
    final_df = pd.DataFrame(matched_records)
    
    # Drop the temporary "_set" columns to keep your data clean
    columns_to_drop = ['author_name_set', 'nama_usaha_set', 'nama_komersial_set']
    final_df = final_df.drop(columns=[col for col in columns_to_drop if col in final_df.columns])
    
    print(f"[DEBUG] Finished! Final Dataset has {len(final_df)} rows (including duplicates).")
    return final_df

# To run it:
# final_identified_data = run_entity_resolution(data_clean, existing_data)

In [5]:
# 1. Load datasets 
# batch 1
data_1 = pd.read_csv('../storage/batch 1/seller_candidates_20260418_010245.csv')
data_2 = pd.read_csv('../storage/batch 1/seller_candidates_20260419_014612.csv')
data_3 = pd.read_csv('../storage/batch 1/seller_candidates_20260421_220819.csv')
data_4 = pd.read_csv('../storage/batch 1/seller_candidates_20260424_233535.csv')
data_5 = pd.read_csv('../storage/batch 1/seller_candidates_20260429_083343.csv')
data_6 = pd.read_csv('../storage/batch 1/seller_candidates_20260502_064143.csv')
data_7 = pd.read_csv('../storage/batch 1/seller_candidates_20260502_091039.csv')
data_8 = pd.read_csv('../storage/batch 1/seller_candidates.csv')

data = pd.concat([data_1, data_2, data_3, data_4, data_5, data_6, data_7, data_8], ignore_index=True)

print(f"Total records loaded: {len(data)}")
print("Columns in existing data:", data.columns.tolist())



Total records loaded: 1971
Columns in existing data: ['source_name', 'source_url', 'post_id', 'post_url', 'author_name', 'phone', 'post_text', 'timestamp_text', 'scraped_at', 'extraction_method', 'extraction_method;']


In [6]:
# Load existing master database for comparison
existing_data = pd.read_excel('masteri-direktori.xlsx',sheet_name='Sheet1')
print(f"Existing master records loaded: {len(existing_data)}")
print("Columns in existing data:", existing_data.columns.tolist())
# load previous batches for comparison


Existing master records loaded: 85702
Columns in existing data: ['idsbr', 'nama_usaha', 'nama_komersial_usaha', 'alamat', 'nama_sls', 'kodepos', 'nomor_telepon', 'nomor_whatsapp', 'email', 'website', 'latitude', 'longitude', 'keberadaan_usaha', 'idsbr_master', 'kdprov_pindah', 'kdkab_pindah', 'kdprov', 'kdkab', 'kdkec', 'kddesa', 'jenis_kepemilikan_usaha', 'bentuk_badan_hukum_usaha', 'deskripsi_badan_usaha_lainnya', 'tahun_berdiri', 'jaringan_usaha', 'sektor_institusi', 'deskripsi_kegiatan_usaha', 'kategori', 'kbli', 'produk_usaha', 'sumber_profiling', 'catatan_profiling']


# Data cleaning and transformation

In [7]:
# 2. change phone number format to 62 format
# If the data in various formats, convert to 62 format
data['phone'] = normalize_to_62(data['phone'])
existing_data['nomor_telepon'] = normalize_to_62(existing_data['nomor_telepon'])
existing_data['nomor_whatsapp'] = normalize_to_62(existing_data['nomor_whatsapp'])

In [8]:
print(data['phone'].head())
print(existing_data['nomor_telepon'].head())

0    6285768582236
1    6285783882272
2    6285783882272
3              NaN
4    6285789496358
Name: phone, dtype: str
0    6242821956
1    6242821455
2           NaN
3           NaN
4           NaN
Name: nomor_telepon, dtype: str


In [9]:
# drop any NaN values and duplicates in the phone column before comparison
data_clean = data.dropna(subset=['phone']).drop_duplicates(subset=['phone'])
print(f"Records with valid phone numbers: {len(data_clean)}")

Records with valid phone numbers: 401


In [10]:
# 1. Generate the Stop Words using BOTH master name columns
# We look at both legal and commercial names to get a perfect list of junk words.
master_name_columns = ['nama_usaha', 'nama_komersial_usaha']
dynamic_stop_words = generate_dynamic_stop_words(existing_data, master_name_columns, top_n=15)
print(f"\n[DEBUG] Dynamic Stop Words Set: {dynamic_stop_words}")

# 2. Apply the cleaner to the Master Data (existing_data)
# We create two new temporary columns that just hold the clean "Sets" of words
existing_data['nama_usaha_set'] = existing_data['nama_usaha'].apply(
    lambda x: create_name_set(x, dynamic_stop_words)
)
existing_data['nama_komersial_set'] = existing_data['nama_komersial_usaha'].apply(
    lambda x: create_name_set(x, dynamic_stop_words)
)

# 3. Apply the cleaner to the Scraped Data (data_clean)
data_clean['author_name_set'] = data_clean['author_name'].apply(
    lambda x: create_name_set(x, dynamic_stop_words)
)


[DEBUG] Top 15 Dynamic Stop Words Generated:
  - jual: 15026 times
  - usaha: 9788 times
  - campuran: 5551 times
  - industri: 4649 times
  - gula: 3628 times
  - penjual: 3096 times
  - merah: 3033 times
  - hj: 2948 times
  - menjual: 2816 times
  - kios: 2491 times
  - toko: 2413 times
  - bensin: 2157 times
  - beli: 1992 times
  - ikan: 1810 times
  - batu: 1629 times

[DEBUG] Dynamic Stop Words Set: {'campuran', 'toko', 'merah', 'industri', 'hj', 'kios', 'penjual', 'batu', 'beli', 'gula', 'menjual', 'bensin', 'usaha', 'jual', 'ikan'}


# Compare scraped data with master data

In [11]:
# print all columns name for comparison
print("Columns in data_clean:", data_clean.columns.tolist())
print("Columns in existing_data:", existing_data.columns.tolist())

Columns in data_clean: ['source_name', 'source_url', 'post_id', 'post_url', 'author_name', 'phone', 'post_text', 'timestamp_text', 'scraped_at', 'extraction_method', 'extraction_method;', 'author_name_set']
Columns in existing_data: ['idsbr', 'nama_usaha', 'nama_komersial_usaha', 'alamat', 'nama_sls', 'kodepos', 'nomor_telepon', 'nomor_whatsapp', 'email', 'website', 'latitude', 'longitude', 'keberadaan_usaha', 'idsbr_master', 'kdprov_pindah', 'kdkab_pindah', 'kdprov', 'kdkab', 'kdkec', 'kddesa', 'jenis_kepemilikan_usaha', 'bentuk_badan_hukum_usaha', 'deskripsi_badan_usaha_lainnya', 'tahun_berdiri', 'jaringan_usaha', 'sektor_institusi', 'deskripsi_kegiatan_usaha', 'kategori', 'kbli', 'produk_usaha', 'sumber_profiling', 'catatan_profiling', 'nama_usaha_set', 'nama_komersial_set']


In [12]:
final_identified_data = run_entity_resolution(data_clean, existing_data)

[DEBUG] Starting Entity Resolution for 401 scraped records...
[DEBUG] Finished! Final Dataset has 8048 rows (including duplicates).


In [13]:
with pd.ExcelWriter("result/batch_1.xlsx") as writer:
    data_clean.to_excel(writer, sheet_name="scraped_data")
    final_identified_data.to_excel(writer, sheet_name="identified_data")